In [ ]:
# --- Imports ---

In [ ]:
import torch

In [ ]:
from torch.utils.data import DataLoader

In [ ]:
from torchvision import transforms

In [ ]:
import pandas as pd

In [ ]:
import numpy as np

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
from src.dataset import PapyriDataset

In [ ]:
from src.model import create_resnet_regressor

In [ ]:
from src.utils import save_predictions_plot

In [ ]:
# --- Setup ---

In [ ]:
import torch

# --- Device Setup ---
# This code block will automatically select the best available hardware.
# 1. NVIDIA GPU (CUDA) - For your Windows PC
# 2. Apple Silicon GPU (Metal Performance Shaders) - For your MacBook
# 3. CPU - Fallback

if torch.cuda.is_available():
    device = torch.device("cuda")
    print(f"✅ Using device: {torch.cuda.get_device_name(0)} (NVIDIA CUDA)")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    print("✅ Using device: Apple Metal (MPS)")
else:
    device = torch.device("cpu")
    print("⚠️ Using device: CPU (Training will be very slow)")

# No other code in your notebooks needs to change.
# All .to(device) calls will now work on any machine.

In [ ]:
MODEL_PATH = 'checkpoints/best_model.pth'

In [ ]:
# Use the *validation* transforms for testing

In [ ]:
test_transform = transforms.Compose([ transforms.Resize((256, 256)), transforms.CenterCrop(224), transforms.ToTensor(), transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]) ])

In [ ]:
# --- Load Test Data ---

In [ ]:
test_dataset = PapyriDataset(csv_file='data/test.csv', img_dir='data/images', transform=test_transform)

In [ ]:
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [ ]:
# --- Load Model ---

In [ ]:
model = create_resnet_regressor(pretrained=False) # Not loading pre-trained weights, just architecture

In [ ]:
model.load_state_dict(torch.load(MODEL_PATH))

In [ ]:
model.to(device)

In [ ]:
model.eval()

In [ ]:
# --- Run Inference ---

In [ ]:
all_labels = []

In [ ]:
all_preds = []

In [ ]:
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        all_preds.extend(outputs.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

In [ ]:
all_labels = np.array(all_labels).flatten()

In [ ]:
all_preds = np.array(all_preds).flatten()

In [ ]:
# --- Calculate Metrics ---

In [ ]:
mae = mean_absolute_error(all_labels, all_preds)

In [ ]:
rmse = np.sqrt(mean_squared_error(all_labels, all_preds))

In [ ]:
print(f"--- Test Set Performance ---")

In [ ]:
print(f"Mean Absolute Error (MAE): {mae:.2f} years")

In [ ]:
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} years")

In [ ]:
# --- Save Results ---

In [ ]:
save_predictions_plot(all_labels, all_preds)

In [ ]:
test_df = pd.read_csv('data/test.csv')

In [ ]:
test_df['predicted_date'] = all_preds

In [ ]:
test_df.to_csv('results/test_predictions.csv', index=False)

In [ ]:
print("Predictions saved to results/test_predictions.csv")